# General overview
Log transofrming and vice versa yielded a very low squared, so we need to do sensitivity analysis to check for further discrepancies in the data.


1. Load data, define features/target (same as before)

2. Set up RepeatedKFold — this is the tool that creates multiple train/test splits automatically

3. For each split, build and train two versions of each model: original-scale and log-transformed

4. Record R² for both versions, every split

5. Average the results and compare

## Step 1: Load Data/ Define Features and target


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score

RANDOM_STATE = 42

# Load data fresh
df = pd.read_csv("final_extract_dataset.csv")

feature_cols = [
    'Region_Code', 'Median_GDP', 'Median_Population_Density',
    'Median_Urban_pct', 'Median_Sanitation_Access_pct',
    'Median_Rainfall', 'Median_Temp'
]

X = df[feature_cols]
y = df['Median_Malaria_Cases']
#y = np.log1p(df['Median_Malaria_Cases'])

## Step 2: set up repeated kfold

In [2]:
# This creates the "repeated splitting" tool
rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)

print("Total number of train/test cycles:", rkf.get_n_splits())
#Ms Maureen said we needed the datat to be tested with different configurations of train 
# and test data. this yields 25 different train test experiments to see if the r^2 really is
# low across all or just one split we did in notebook 2 

Total number of train/test cycles: 25


# Step 3 build and train 2 versions per model

In [3]:
# --- Preprocessing setup (reused inside every fold) ---
categorical_cols = ['Region_Code']
numeric_cols = [
    'Median_GDP', 'Median_Population_Density', 'Median_Urban_pct',
    'Median_Sanitation_Access_pct', 'Median_Rainfall', 'Median_Temp'
]

def build_preprocessor():
    return ColumnTransformer(transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ])

"""
this is wrapped in a function (build_preprocessor()) instead of being one 
fixed object like before. Why? Because we're about to train models 25 times over, on 25 different train/test splits — each fold needs its own fresh, independently-fitted preprocessor, so nothing "leaks" information between folds. Calling this function each time gives us a brand-new, untrained ColumnTransformer for every fold.
"""

'\nthis is wrapped in a function (build_preprocessor()) instead of being one \nfixed object like before. Why? Because we\'re about to train models 25 times over, on 25 different train/test splits — each fold needs its own fresh, independently-fitted preprocessor, so nothing "leaks" information between folds. Calling this function each time gives us a brand-new, untrained ColumnTransformer for every fold.\n'

In [4]:
alphas_wide = np.logspace(-3, 6, 50)  # slightly fewer alphas than before, since this runs 25x now
#I reduced the alpha search from 100 to 50 candidates (np.logspace(-3, 6, 50) instead of 100) purely for speed — with 25 folds × 8 models × alpha search each, this could get slow otherwise. 50 candidates is still plenty for a reliable search.

def build_models():
    """Returns a fresh dictionary of un-trained models, both original and log-transformed versions."""
    
    original_models = {
        'MLR': LinearRegression(),
        'Ridge': RidgeCV(alphas=alphas_wide, cv=5),
        'Lasso': LassoCV(alphas=alphas_wide, cv=5, random_state=RANDOM_STATE, max_iter=10000),
        'ElasticNet': ElasticNetCV(alphas=alphas_wide, cv=5, random_state=RANDOM_STATE, max_iter=10000),
    }
    
    log_models = {
        'MLR_log': TransformedTargetRegressor(LinearRegression(), func=np.log1p, inverse_func=np.expm1),
        'Ridge_log': TransformedTargetRegressor(RidgeCV(alphas=alphas_wide, cv=5), func=np.log1p, inverse_func=np.expm1),
        'Lasso_log': TransformedTargetRegressor(LassoCV(alphas=alphas_wide, cv=5, random_state=RANDOM_STATE, max_iter=10000), func=np.log1p, inverse_func=np.expm1),
        'ElasticNet_log': TransformedTargetRegressor(ElasticNetCV(alphas=alphas_wide, cv=5, random_state=RANDOM_STATE, max_iter=10000), func=np.log1p, inverse_func=np.expm1),
    }
    
    return {**original_models, **log_models}

## Step 4: Record all the R^2 for all models and their log twins

In [5]:
# This will store every single result: which model, which split, what R² it got
all_results = []

fold_number = 0

for train_idx, test_idx in rkf.split(X):
    fold_number += 1
    
    # Split the data for this specific fold
    X_train_fold = X.iloc[train_idx]
    X_test_fold = X.iloc[test_idx]
    y_train_fold = y.iloc[train_idx]
    y_test_fold = y.iloc[test_idx]
    
    # Get a fresh set of untrained models for this fold
    models = build_models()
    
    for model_name, model in models.items():
        # Build a fresh pipeline: fresh preprocessor + this model
        pipe = Pipeline([
            ('preprocessor', build_preprocessor()),
            ('regressor', model)
        ])
        
        # Train on this fold's training data
        pipe.fit(X_train_fold, y_train_fold)
        
        # Predict on this fold's test data
        preds = pipe.predict(X_test_fold)
        
        # Score it
        r2 = r2_score(y_test_fold, preds)
        
        # Record the result
        all_results.append({
            'fold': fold_number,
            'model': model_name,
            'r2': r2
        })

print(f"Done. Total results recorded: {len(all_results)}")

Done. Total results recorded: 200


## Step 5: Average results for comparison

In [6]:
results_df = pd.DataFrame(all_results)

# Average R² and consistency (standard deviation) per model, across all 25 folds
summary = results_df.groupby('model')['r2'].agg(['mean', 'std', 'min', 'max']).round(4)
summary = summary.sort_values('mean', ascending=False)

summary

,mean,std,min,max
model,,,,
Lasso,0.5603,0.1545,0.0183,0.7787
ElasticNet,0.5591,0.1413,0.1831,0.7791
Ridge,0.5590,0.1457,0.1417,0.7765
MLR,0.5565,0.1625,0.0107,0.7702
ElasticNet_log,0.5099,0.2123,-0.2960,0.7633
Ridge_log,0.5098,0.2025,-0.2423,0.7508
MLR_log,0.5097,0.2189,-0.3455,0.7457
Lasso_log,0.5091,0.2099,-0.2724,0.7513


This repeated cross-validation is showing us that performance is highly unstable depending on which 20 countries happen to land in our test set. With only 99 countries and likely a handful of extreme outlier countries (very high case counts), a single split can randomly put those outliers all in train (making test look easy) or all in test (making the model look terrible) — and our original split happened to be a relatively favorable one.

A repeated 5×5-fold cross-validation revealed that model performance is highly sensitive to sample composition, with wide variance in R² across folds (std often exceeding the mean). Contrary to the initial single-split evaluation, log-transformed target models consistently outperformed their untransformed counterparts across repeated folds (e.g., Lasso_log mean R²=-0.279 vs Lasso mean R²=-0.477), suggesting the transformation does provide meaningful, consistent benefit despite both remaining below acceptable predictive thresholds. This instability likely stems from the small sample size (n=99) combined with a small number of extreme outlier countries, which disproportionately affect performance depending on their allocation to train or test folds in any given split.